# 03 — Walk-Forward Calibration : apprendre les poids par régression

**Objectif** : remplacer les constantes choisies arbitrairement (`SENTIMENT_WEIGHT=0.5`, `MOMENTUM_WEIGHT=0.5`, `SIGNAL_STRENGTH=0.20`) par des **coefficients appris sur les données**, sans look-ahead bias.

**Approche** : régression Ridge walk-forward
- Fenêtre d'entraînement glissante : 40 jours
- Fenêtre de test : 1 jour (prédiction du rendement J+1)
- Features : `[sentiment_score, momentum_tanh_zscore, sentiment × momentum]`
- Cible : rendement J+1 par ticker
- Les coefficients appris remplacent les constantes fixes dans `mu_adjusted`

**Convention anti-look-ahead** : à chaque jour J, le modèle est entraîné **uniquement sur J-40 à J-1**. Aucune donnée de J ou au-delà n'entre dans l'entraînement.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from pypfopt import EfficientFrontier, expected_returns, risk_models

TICKERS = ['NKE', 'TGT', 'DIS', 'SBUX', 'TSLA']
DATA_INPUT  = ROOT / 'backtesting_2' / 'data_input'
DATA_OUTPUT = ROOT / 'backtesting_2' / 'data_output'
COLORS = {'NKE': '#1f77b4', 'TGT': '#ff7f0e', 'DIS': '#2ca02c', 'SBUX': '#d62728', 'TSLA': '#9467bd'}
print("Imports OK")

## 1. Charger les scores et les prix

In [ ]:
sent_df = pd.read_csv(DATA_OUTPUT / 'daily_sentiment_scores.csv', parse_dates=['date'])
mom_df  = pd.read_csv(DATA_OUTPUT / 'daily_momentum_scores.csv',  parse_dates=['date'])

sent_piv = sent_df.pivot(index='date', columns='ticker', values='sentiment_score')[TICKERS]
raw_piv  = mom_df.pivot( index='date', columns='ticker', values='momentum_raw' if 'momentum_raw' in mom_df.columns
                          else 'mom_1m')[TICKERS]

# Recalcul momentum_raw depuis mom_1m + mom_3m
mom_df['momentum_raw'] = mom_df[['mom_1m', 'mom_3m']].mean(axis=1)
raw_piv = mom_df.pivot(index='date', columns='ticker', values='momentum_raw')[TICKERS]

# Normalisation tanh(z-score) pour le momentum
def tanh_zscore_row(row):
    mu, sigma = row.mean(), row.std()
    if sigma < 1e-10:
        return row * 0
    return np.tanh((row - mu) / sigma)

mom_tanh_piv = raw_piv.apply(tanh_zscore_row, axis=1)

# Prix et rendements
prices = yf.download(TICKERS, start='2025-10-01', end='2026-06-03',
                     auto_adjust=True, progress=False)['Close'][TICKERS]
returns = prices.pct_change()

# Rendement J+1 (target)
fwd_returns = returns.shift(-1)

# Jours communs
common_dates = sent_piv.index.intersection(mom_tanh_piv.index).intersection(fwd_returns.index)
common_dates = common_dates.sort_values()
print(f"Dates communes : {common_dates[0].date()} → {common_dates[-1].date()} ({len(common_dates)} jours)")

## 2. Construction des features

Pour chaque jour J et chaque ticker, les features sont :
- `sentiment` : score FinBERT EMA
- `momentum` : tanh(z-score) du momentum 1m/3m
- `sent_x_mom` : interaction croisée (capte les signaux cohérents)

In [ ]:
records = []
for date in common_dates:
    for t in TICKERS:
        s   = sent_piv.loc[date, t]     if date in sent_piv.index    else 0.0
        m   = mom_tanh_piv.loc[date, t] if date in mom_tanh_piv.index else 0.0
        fwd = fwd_returns.loc[date, t]  if date in fwd_returns.index  else np.nan
        records.append({
            'date': date, 'ticker': t,
            'sentiment': s, 'momentum': m, 'sent_x_mom': s * m,
            'fwd_return': fwd
        })

panel = pd.DataFrame(records).dropna(subset=['fwd_return'])
print(f"Panel : {len(panel)} observations, {panel['date'].nunique()} jours")
panel.head(10)

## 3. Walk-forward Ridge — apprendre les poids jour après jour

In [ ]:
TRAIN_WINDOW = 40   # jours d'historique pour entraîner
ALPHA_RIDGE  = 0.1  # régularisation Ridge (évite l'overfitting sur petit échantillon)
FEATURE_COLS = ['sentiment', 'momentum', 'sent_x_mom']

all_dates = sorted(panel['date'].unique())

# Résultats walk-forward
wf_results = []  # {date, ticker, pred_signal, actual_coef_sent, actual_coef_mom}
coef_history = []  # évolution des coefficients dans le temps

print(f"Walk-forward : {len(all_dates) - TRAIN_WINDOW} étapes de calibration")

for i in range(TRAIN_WINDOW, len(all_dates)):
    train_dates = all_dates[i - TRAIN_WINDOW : i]   # [J-40, J-1]
    test_date   = all_dates[i]                        # J

    train_panel = panel[panel['date'].isin(train_dates)]
    test_panel  = panel[panel['date'] == test_date]

    if len(train_panel) < 10 or len(test_panel) == 0:
        continue

    X_train = train_panel[FEATURE_COLS].values
    y_train = train_panel['fwd_return'].values
    X_test  = test_panel[FEATURE_COLS].values

    # Pipeline : StandardScaler + Ridge
    model = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge',  Ridge(alpha=ALPHA_RIDGE))
    ])
    model.fit(X_train, y_train)

    pred_signals = model.predict(X_test)
    coefs = model.named_steps['ridge'].coef_

    coef_history.append({'date': test_date, 'c_sentiment': coefs[0], 'c_momentum': coefs[1], 'c_interaction': coefs[2]})

    for j, row in enumerate(test_panel.itertuples()):
        wf_results.append({
            'date': test_date,
            'ticker': row.ticker,
            'pred_signal': pred_signals[j],
            'actual_fwd': row.fwd_return
        })

wf_df   = pd.DataFrame(wf_results)
coef_df = pd.DataFrame(coef_history)
print(f"Walk-forward terminé : {len(coef_df)} jours calibrés")

## 4. Évolution des coefficients appris

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(coef_df['date'], coef_df['c_sentiment'],    label='Coefficient sentiment',   color='#e74c3c', lw=2)
ax.plot(coef_df['date'], coef_df['c_momentum'],     label='Coefficient momentum',    color='#3498db', lw=2)
ax.plot(coef_df['date'], coef_df['c_interaction'],  label='Coefficient interaction', color='#9b59b6', lw=1.5, ls='--')
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_title('Évolution des coefficients Ridge appris (fenêtre glissante 40j)', fontsize=12)
ax.set_ylabel('Coefficient (contribution au rendement J+1 prédit)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))
ax.legend()
ax.grid(alpha=0.2)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print("\nMoyennes des coefficients sur toute la période :")
print(coef_df[['c_sentiment', 'c_momentum', 'c_interaction']].describe().round(6))

## 5. Backtest walk-forward vs stratégie fixe vs equal-weight

In [ ]:
backtest_dates_wf = sorted(wf_df['date'].unique())
WEIGHT_BOUNDS = (0.05, 0.40)
RISK_AVERSION = 1.0

def run_backtest_from_signals(signal_piv, label, signal_strength=0.20):
    capital = 100_000.0
    portfolio_values = [capital]

    for day in backtest_dates_wf[:-1]:
        train_prices = prices.loc[:day].dropna(how='all')
        if len(train_prices) < 65:
            next_day = backtest_dates_wf[backtest_dates_wf.index(day) + 1]
            ret = sum(1/5 * (prices.loc[next_day, t] / prices.loc[day, t] - 1) for t in TICKERS)
            portfolio_values.append(portfolio_values[-1] * (1 + ret))
            continue

        try:
            mu_hist = expected_returns.mean_historical_return(train_prices, compounding=True, frequency=252)
            if day in signal_piv.index:
                sig = signal_piv.loc[day].reindex(mu_hist.index).fillna(0)
            else:
                sig = pd.Series(0.0, index=mu_hist.index)
            mu_adj = mu_hist + signal_strength * sig
            S = risk_models.CovarianceShrinkage(train_prices, frequency=252).ledoit_wolf()
            ef = EfficientFrontier(mu_adj, S, weight_bounds=WEIGHT_BOUNDS)
            ef.max_quadratic_utility(risk_aversion=RISK_AVERSION)
            w = dict(ef.clean_weights())
        except Exception:
            w = {t: 1/5 for t in TICKERS}

        idx = backtest_dates_wf.index(day)
        next_day = backtest_dates_wf[idx + 1]
        port_ret = sum(w.get(t, 0) * (prices.loc[next_day, t] / prices.loc[day, t] - 1) for t in TICKERS)
        portfolio_values.append(portfolio_values[-1] * (1 + port_ret))

    return pd.Series(portfolio_values, index=backtest_dates_wf[:len(portfolio_values)])

# Signal walk-forward : pred_signal comme proxy du global_score
wf_signal_piv = wf_df.pivot(index='date', columns='ticker', values='pred_signal')

# Signal fixe (actuel) : sent 0.5 + mom 0.5 avec tanh-zscore
fixed_global_piv = 0.5 * sent_piv.reindex(backtest_dates_wf) + 0.5 * mom_tanh_piv.reindex(backtest_dates_wf)

print("Backtest walk-forward en cours...")
vals_wf    = run_backtest_from_signals(wf_signal_piv,    'Walk-Forward Ridge', signal_strength=5.0)  # pred_signal en ordre de rendement
vals_fixed = run_backtest_from_signals(fixed_global_piv, 'Signal fixe 0.5/0.5', signal_strength=0.20)

ew_ret_wf = prices.loc[backtest_dates_wf].pct_change().fillna(0).mean(axis=1)
vals_ew_wf = (1 + ew_ret_wf).cumprod() * 100_000

print("Done.")

In [ ]:
def sharpe(series, freq=252):
    r = series.pct_change().dropna()
    return float(r.mean() / r.std() * np.sqrt(freq)) if r.std() > 0 else 0

def max_dd(series):
    return float(((series - series.cummax()) / series.cummax()).min())

fig, axes = plt.subplots(2, 1, figsize=(13, 10))

ax = axes[0]
for label, vals, color in [
    ('Walk-Forward Ridge (nouveau)', vals_wf,    '#2ecc71'),
    ('Signal fixe 0.5/0.5 (actuel)', vals_fixed, '#e74c3c'),
    ('Equal-weight',                  vals_ew_wf, '#f39c12'),
]:
    ax.plot(vals.index, vals, label=label, lw=2, color=color)

ax.axhline(100_000, color='gray', lw=0.8, ls='--')
ax.set_title('Walk-Forward Ridge vs Signal fixe vs Equal-weight')
ax.set_ylabel('Valeur du portefeuille ($)')
ax.legend()
ax.grid(alpha=0.2)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%d/%m'))

ax2 = axes[1]
ax2.axis('off')
rows_tbl = []
for label, vals in [
    ('Walk-Forward Ridge', vals_wf),
    ('Signal fixe 0.5/0.5', vals_fixed),
    ('Equal-weight', vals_ew_wf),
]:
    rows_tbl.append([
        label,
        f"{(vals.iloc[-1]/vals.iloc[0]-1)*100:+.2f}%",
        f"{sharpe(vals):.2f}",
        f"{100*max_dd(vals):.2f}%",
        f"${vals.iloc[-1]:,.0f}"
    ])

tbl = ax2.table(
    cellText=rows_tbl,
    colLabels=['Stratégie', 'Rendement', 'Sharpe', 'Max DD', 'Capital final'],
    loc='center', cellLoc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1, 2)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 6. Interprétation & recommandations

### Ce que le walk-forward apprend
Les coefficients Ridge révèlent **empiriquement** le poids relatif du sentiment et du momentum pour prédire le rendement J+1 sur cet univers de 5 tickers.

### Limites à garder en tête
- **N=5 tickers × ~40 jours = ~200 observations d'entraînement** : très petit. La régularisation Ridge (α=0.1) atténue l'overfitting mais ne l'élimine pas.
- Les coefficients peuvent être instables d'une fenêtre à l'autre (voir graphique section 4).
- **Ce n'est pas un oracle** : les corrélations sur 40 jours ne garantissent pas la prédictivité sur les 40 jours suivants.

### Prochaines étapes suggérées
1. Corriger d'abord la normalisation (Notebook 02) — c'est le gain le plus fiable
2. Utiliser les coefficients moyens de la section 5 pour recalibrer `SENTIMENT_WEIGHT`, `MOMENTUM_WEIGHT` et `SIGNAL_STRENGTH` dans `config_backtest.py`
3. Sur 1 an de données (si disponible), relancer ce walk-forward pour confirmer la stabilité